<a href="https://colab.research.google.com/github/diyanrahma/Retrieval-Augmented-Generation-Automatic-Fact-Checking-PubHealth-Dataset/blob/main/RAG_Pertanyaan_Tidak_Normalisasi_Claim_Tidak_Normalisasi.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Instal Library

In [ ]:
!pip install -q sentence-transformers faiss-cpu transformers datasets tqdm scikit-learn pandas

In [ ]:
import os, re, numpy as np, pandas as pd, torch, faiss
from tqdm.auto import tqdm
from datasets import load_dataset
from sentence_transformers import SentenceTransformer, CrossEncoder
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from IPython.display import display

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("DEVICE:", DEVICE)

# Load Dataset

In [ ]:
import pandas as pd
import re

df = pd.read_excel("/content/pubhealth_biomed_only.xlsx")

def normalize_text(x):
    if not isinstance(x, str):
        x = str(x)
    x = re.sub(r'\s+', ' ', x).strip()
    x = re.sub(r'\"+', '', x)
    return x

print("Kolom tersedia:", df.columns.tolist())
print("Jumlah per Label:", df["label"].value_counts().to_dict())
df.head()

In [ ]:
Q_PATH = "/content/pertanyaan tanpa normalisasi.xlsx"
qdf = pd.read_excel(Q_PATH)
qdf.columns = [c.strip().lower() for c in qdf.columns]

need = {"id","question","label","question_type"}
missing = need - set(qdf.columns)
if missing:
    raise ValueError(f"Kolom wajib hilang: {missing}. Pastikan ada: {need}")

qdf = qdf[qdf["id"].astype(str).isin(df["id"].astype(str))].reset_index(drop=True)

qdf["gold_label"] = qdf["label"].astype(str).str.lower()
display(qdf.head())

In [ ]:
before = qdf["label"].astype(str).str.len()
after  = qdf["gold_label"].astype(str).str.len()

changed = (before != after)

print("Jumlah baris yang berubah:", changed.sum())
display(qdf.loc[changed, "label"].head())
display(qdf.loc[changed, "gold_label"].head())

# Token & Chunking

In [ ]:
from transformers import AutoTokenizer
from tqdm.auto import tqdm
import pandas as pd

_ref_tok = AutoTokenizer.from_pretrained("google/flan-t5-base")

MIN_TOK = 110
MAX_TOK = 180
OVERLAP_TOK = 30


def chunk_by_tokens_strict(text, min_tokens=MIN_TOK, max_tokens=MAX_TOK, overlap=OVERLAP_TOK):
    tokens = _ref_tok(str(text), truncation=False).input_ids
    total_len = len(tokens)

    chunks = []
    start = 0

    while start < total_len:
        end = start + max_tokens
        chunk_ids = tokens[start:end]

        if len(chunk_ids) < min_tokens and len(chunks) > 0:
            prev_ids = _ref_tok(chunks[-1], truncation=False).input_ids
            merged_ids = prev_ids + chunk_ids
            chunks[-1] = _ref_tok.decode(merged_ids, skip_special_tokens=True)
            break

        chunk_text = _ref_tok.decode(chunk_ids, skip_special_tokens=True)
        chunks.append(chunk_text)

        start += max_tokens - overlap

    return chunks

chunk_rows = []

for _, r in tqdm(df.iterrows(), total=len(df), desc="Building chunks"):
    doc_id = r["id"]
    evidence_text = r["evidence"]

    chs = chunk_by_tokens_strict(evidence_text)

    for j, ch in enumerate(chs):
        chunk_rows.append({
            "doc_id": doc_id,
            "chunk_id": f"{doc_id}::c{j}",
            "chunk_text": ch
        })

df_chunks = pd.DataFrame(chunk_rows)

print("Jumlah dokumen:", len(df))
print("Total chunks:", len(df_chunks))

display(df_chunks.head())

# Embedding + Index (FAISS)

In [ ]:
EMB_MODEL_NAME = "all-MiniLM-L6-v2"
emb_model = SentenceTransformer(EMB_MODEL_NAME, device=DEVICE)

def enc_passages(texts):
    return emb_model.encode(
        [f"passage: {t}" for t in texts],
        batch_size=128,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True
    )

def enc_query(q):
    return emb_model.encode(
        [f"query: {q}"],
        convert_to_numpy=True,
        normalize_embeddings=True
    )

chunk_emb = enc_passages(df_chunks["chunk_text"].tolist())
index = faiss.IndexFlatIP(chunk_emb.shape[1])
index.add(chunk_emb)

def faiss_search_topk(question, topk=100):
    qv = enc_query(question)
    scores, idxs = index.search(qv, topk)
    return scores[0], idxs[0]

# Reranker

In [ ]:
TOPK_RETRIEVE = 100
TOPM_CONTEXT  = 8
USE_RERANKER  = True

if USE_RERANKER:
    reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2", device=DEVICE)

def pick_context(question, faiss_idxs, topm=TOPM_CONTEXT, max_per_doc=2):
    cand = df_chunks.iloc[faiss_idxs][["doc_id","chunk_text"]].copy()
    if USE_RERANKER:
        scores = reranker.predict(list(zip([question]*len(cand), cand["chunk_text"].tolist())))
        cand["score"] = scores
        cand = cand.sort_values("score", ascending=False)
    picked, per_doc = [], {}
    for _, r in cand.iterrows():
        d = r["doc_id"]
        per_doc[d] = per_doc.get(d, 0) + 1
        if per_doc[d] <= max_per_doc:
            picked.append(r["chunk_text"])
        if len(picked) >= topm:
            break
    return picked

In [ ]:
id2gold = (
    df[["id","label"]]
    .assign(label_bool=lambda d: d["label"].map({"true":True,"false":False}))
    .set_index("id")["label_bool"]
    .to_dict()
)

# Generate

In [ ]:
GEN_MODEL = "google/flan-t5-base"
g_tok = AutoTokenizer.from_pretrained(GEN_MODEL)
g_mdl = AutoModelForSeq2SeqLM.from_pretrained(GEN_MODEL).to(DEVICE)

def generate_text(prompt, max_new=12, beams=4, rep_pen=1.2, ngram=2):
    ids = g_tok(prompt, return_tensors="pt", truncation=True).to(DEVICE)
    with torch.no_grad():
        out = g_mdl.generate(
            **ids,
            max_new_tokens=max_new,
            num_beams=beams,
            do_sample=False,
            no_repeat_ngram_size=ngram,
            repetition_penalty=rep_pen
        )
    return g_tok.decode(out[0], skip_special_tokens=True).strip()

YESNO_REGEX = r"^(is|are|was|were|do|does|did|can|could|should|would|will|has|have|had|may|might|must)\b"
WH_REGEX    = r"^(what|why|when|where|who|whom|whose|which|how)\b"
def is_yesno(q): return bool(re.match(YESNO_REGEX, q.strip().lower()))
def is_wh(q):    return bool(re.match(WH_REGEX,    q.strip().lower()))

def answer_yesno(question, evidences):
    if not evidences: return "NO"
    ctx = "\n\n---\n\n".join(evidences)
    prompt = (
        "You are a medical fact-checking assistant.\n"
        "Use ONLY the evidence below to answer YES or NO.\n"
        "If evidence is insufficient, answer 'NO'.\n\n"
        f"EVIDENCE:\n{ctx}\n\nQUESTION:\n{question}\n\nANSWER:"
    )
    out = generate_text(prompt, max_new=3, beams=6)
    s = str(out).strip().upper()
    if "YES" in s: return "YES"
    if "NO"  in s: return "NO"
    return "NO"

def answer_wh(question, evidences, max_words=15):
    if not evidences:
        return "UNKNOWN"

    ctx = "\n\n---\n\n".join(evidences)

    prompt = (
        "You are a medical fact-checking assistant.\n"
        f"Answer concisely (<= {max_words} words) based only on the evidence below. "
        "If the answer truly cannot be determined from the evidence, respond with 'UNKNOWN'.\n\n"
        f"EVIDENCE:\n{ctx}\n\nQUESTION:\n{question}\n\nANSWER:"
    )

    out = generate_text(prompt, max_new=24, beams=6)

    ans = re.sub(r"\s+", " ", out).strip()

    if not ans or ans.upper() == "UNKNOWN":
        return "UNKNOWN"

    return " ".join(ans.split()[:max_words])

# Retreieve FAISS + hasil rerank

In [ ]:
def retrieve_candidates(question, topk=TOPK_RETRIEVE):
    _, idxs = faiss_search_topk(question, topk)
    items = []
    for i in idxs:
        row = df_chunks.iloc[int(i)]
        items.append((row["chunk_text"], row["doc_id"]))
    return items

def rerank_passages(question, candidates, topm=TOPM_CONTEXT, max_per_doc=2):
    passages = [p for (p, _) in candidates]
    if USE_RERANKER:
        scores = reranker.predict(list(zip([question]*len(passages), passages)))
        tmp = pd.DataFrame({
            "passage": passages,
            "doc_id": [pid for (_, pid) in candidates],
            "score": scores
        }).sort_values("score", ascending=False)
    else:
        tmp = pd.DataFrame({
            "passage": passages,
            "doc_id": [pid for (_, pid) in candidates]
        })
        tmp["score"] = np.arange(len(tmp))[::-1]

    picked, per_doc = [], {}
    for _, r in tmp.iterrows():
        d = r["doc_id"]
        per_doc[d] = per_doc.get(d, 0) + 1
        if per_doc[d] <= max_per_doc:
            picked.append((r["passage"], d, float(r.get("score", 0.0))))
        if len(picked) >= topm:
            break
    return picked

# Label Fakta & Mitos + Explanation

In [ ]:
def yn_to_fm(yn: str) -> str:
    s = str(yn).strip().upper()
    if "YES" in s: return "FAKTA"
    if "NO"  in s: return "MITOS"
    return "MITOS"

_SENT_SPLIT = re.compile(r'(?<=[.!?])\s+')

def split_sentences(text: str):
    sents = _SENT_SPLIT.split(str(text).strip())
    return [s.strip() for s in sents if s.strip()]

def extract_support_snippets(question: str, picked, max_sentences: int = 2, max_chars: int = 360):
    """
    picked: list of (passage, doc_id, score) dari rerank_passages()
    Mengembalikan 1–2 kalimat paling relevan (berdasarkan CrossEncoder).
    """
    top_passages = [p for (p, _, _) in sorted(picked, key=lambda x: -x[2])[:3]]
    candidate_sents = []
    for p in top_passages:
        for s in split_sentences(p):
            if len(s.split()) >= 5:
                candidate_sents.append(s)

    if not candidate_sents:
        return ""

    pairs = list(zip([question]*len(candidate_sents), candidate_sents))
    scores = reranker.predict(pairs) if USE_RERANKER else np.arange(len(candidate_sents))[::-1]
    order = np.argsort(scores)[::-1]
    chosen, seen = [], set()
    for idx in order:
        s = candidate_sents[idx]
        key = re.sub(r'\W+', ' ', s.lower())[:80]
        if key in seen:
            continue
        chosen.append(s)
        seen.add(key)
        if len(chosen) >= max_sentences:
            break

    expl = " ".join(chosen)
    if len(expl) > max_chars:
        expl = expl[:max_chars].rstrip() + "..."
    return expl

def answer_yesno_with_explanation(question, evidences, picked_for_support):
    """
    Pakai answer_yesno() → YES/NO → FAKTA/MITOS + ambil 1–2 kalimat support.
    Tapi explanation hanya menampilkan bukti (tanpa awalan 'FAKTA — Bukti:')
    """
    yn = answer_yesno(question, evidences)
    fm = yn_to_fm(yn)
    support = extract_support_snippets(question, picked_for_support, max_sentences=2, max_chars=360)

    explanation = support if support else "Bukti tidak cukup spesifik pada kutipan."

    return fm, yn, explanation

# Output

In [ ]:
USE_SAMPLE_100 = False
if USE_SAMPLE_100:
    q_eval = qdf.sample(100, random_state=42).reset_index(drop=True)
else:
    q_eval = qdf.copy().reset_index(drop=True)

rows = []
for q in tqdm(q_eval["question"].tolist(), desc="RAG answering"):
    cand = retrieve_candidates(q)
    top_items = rerank_passages(q, cand)
    evidences = [p for (p, _, _) in top_items]
    parent_ids = [pid for (_, pid, _) in top_items]

    if parent_ids:
        labs = [id2gold.get(pid, False) for pid in parent_ids]
        label_bool = (labs.count(True) >= labs.count(False))
        label = "True" if label_bool else "False"
    else:
        label = "False"

    if is_yesno(q):
        ans_disp, ans_raw, explanation = answer_yesno_with_explanation(q, evidences, top_items)
    elif is_wh(q):
        ans_disp = answer_wh(q, evidences, max_words=8)
        ans_raw  = ans_disp
        explanation = "—"
    else:
        ans_disp = answer_wh(q, evidences, max_words=8)
        ans_raw  = ans_disp
        explanation = "—"

    rows.append({
        "question": q,
        "answer": ans_disp,
        "explanation": explanation,
        "context_used": "\n\n---\n\n".join(evidences)[:3000],
        "label": label
         })

df_rag_out = pd.DataFrame(rows)
display(df_rag_out.head(10))

# Evaluasi

In [ ]:
df_rag_out = df_rag_out.rename(columns={"question":"question"})

In [ ]:
eval_df = q_eval[["question", "gold_label"]].merge(
    df_rag_out,
    on="question",
    how="inner"
)

In [ ]:
eval_df = eval_df.merge(
    q_eval[["question","question_type"]],
    on="question",
    how="left"
)

eval_df["is_yesno"] = eval_df["question_type"] == "yes/no"

In [ ]:
mask_yesno = eval_df["is_yesno"] & eval_df["answer"].notna()
eval_yn = eval_df[mask_yesno].copy()

print("Jumlah pertanyaan YES/NO untuk evaluasi:", len(eval_yn))

ct = pd.crosstab(eval_yn["gold_label"], eval_yn["answer"])
print("=== Confusion Table (gold_label vs FAKTA/MITOS) ===")
display(ct)

def gold_to_bool(lbl):
    lbl = str(lbl).lower()
    if lbl == "true":
        return True
    if lbl == "false":
        return False
    return None

eval_yn["gold_bool"] = eval_yn["gold_label"].apply(gold_to_bool)

def fm_to_bool(fm):
    if fm == "FAKTA":
        return True
    if fm == "MITOS":
        return False
    return None

eval_yn["pred_bool"] = eval_yn["answer"].apply(fm_to_bool)

mask_tf = eval_yn["gold_bool"].notna() & eval_yn["pred_bool"].notna()
bin_df = eval_yn[mask_tf].copy()

y_true = bin_df["gold_bool"].astype(bool).values
y_pred = bin_df["pred_bool"].astype(bool).values

report = classification_report(
    y_true, y_pred,
    target_names=["False","True"],
    output_dict=True,
    zero_division=0
)

cm = confusion_matrix(y_true, y_pred, labels=[False, True])
acc = accuracy_score(y_true, y_pred)

summary = pd.DataFrame([{
    "Accuracy": acc,
    "Precision(True)": report["True"]["precision"],
    "Recall(True)": report["True"]["recall"],
    "F1(True)": report["True"]["f1-score"],
    "Precision(False)": report["False"]["precision"],
    "Recall(False)": report["False"]["recall"],
    "F1(False)": report["False"]["f1-score"],
}])

print("=== Rangkuman Metrik (TRUE/FALSE saja) ===")
display(summary.style.format("{:.4f}"))

print("\n=== Confusion Matrix (gold TRUE/FALSE saja) ===")
display(pd.DataFrame(cm, index=["gold_False","gold_True"], columns=["pred_False","pred_True"]))

# Non RAG

In [ ]:
def answer_yesno_nonrag(question):
    """
    NON-RAG: Flan-T5 menjawab HANYA YES atau NO.
    Jika pertanyaan bukan yes/no → UNKNOWN.
    """

    if not is_yesno(question):
        return "UNKNOWN"

    prompt = (
        "Answer the following statement ONLY with YES or NO.\n"
        "Do NOT explain. Output must be exactly YES or NO.\n\n"
        f"STATEMENT:\n{question}\n\nANSWER:"
    )

    out = generate_text(prompt, max_new=3, beams=6)
    ans = str(out).strip().upper()

    if ans.startswith("YES"):
        return "YES"
    if ans.startswith("NO"):
        return "NO"

    return "UNKNOWN"

rows_nonrag = []

for _, row in tqdm(q_eval.iterrows(), total=len(q_eval), desc="NON-RAG answering"):

    q = row["question"]
    gold = row["gold_label"]

    yn = answer_yesno_nonrag(q)

    if yn == "YES":
        label_pred = "FAKTA"
    elif yn == "NO":
        label_pred = "MITOS"
    else:
        label_pred = "UNKNOWN"

    rows_nonrag.append({
        "question": q,
        "nonrag_answer": yn,
        "nonrag_label": label_pred,
        "gold_label": gold
    })

df_nonrag_out = pd.DataFrame(rows_nonrag)
display(df_nonrag_out.head(10))

# Evaluasi Non RAG

In [ ]:
rows_nonrag = []

for q in tqdm(q_eval["question"].tolist(), desc="NON-RAG answering"):
    yn = answer_yesno_nonrag(q)

    if yn == "YES":
        label_pred = "FAKTA"
    elif yn == "NO":
        label_pred = "MITOS"
    else:
        label_pred = "UNKNOWN"

    rows_nonrag.append({
        "question": q,
        "nonrag_answer": yn,
        "nonrag_label": label_pred,
        "gold_label": qdf.loc[qdf["question"] == q, "gold_label"].iloc[0]
    })

df_nonrag_out = pd.DataFrame(rows_nonrag)

print(df_nonrag_out.columns)
display(df_nonrag_out.head())

In [ ]:
eval_nonrag = df_nonrag_out.merge(
    q_eval[["question","question_type"]],
    left_on="question",
    right_on="question",
    how="left"
)

eval_nonrag = eval_nonrag[
    eval_nonrag["question_type"].str.strip().str.lower() == "yes/no"
].copy()

print("Jumlah pertanyaan YES/NO:", len(eval_nonrag))

In [ ]:
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score
import pandas as pd

df = eval_nonrag.copy()

df["gold_class"] = df["gold_label"].map({
    "true": "FAKTA",
    "false": "MITOS",
    "mixture": "MIXTURE",
    "unproven": "UNPROVEN"
})

df["pred_class"] = df["nonrag_label"].map({
    "FAKTA": "FAKTA",
    "MITOS": "MITOS",
    "MIXTURE": "UNKNOWN",
    "UNPROVEN": "UNKNOWN",
    "UNKNOWN": "UNKNOWN"
})

df["gold_class"] = df["gold_class"].astype(str).str.upper().str.strip()
df["pred_class"] = df["pred_class"].astype(str).str.upper().str.strip()

gold_labels = ["FAKTA","MITOS","MIXTURE","UNPROVEN"]
pred_labels = ["FAKTA","MITOS","UNKNOWN"]

df = df[
    df["gold_class"].isin(gold_labels) &
    df["pred_class"].isin(pred_labels)
].copy()

cm = confusion_matrix(
    df["gold_class"],
    df["pred_class"],
    labels=gold_labels
)

cm_df = pd.crosstab(
    df["gold_class"],
    df["pred_class"]
).reindex(index=gold_labels, columns=pred_labels, fill_value=0)

print("\n=== Confusion Matrix  ===")
display(cm_df)

accuracy_non = accuracy_score(df["gold_class"], df["pred_class"])
precision_non = precision_score(df["gold_class"], df["pred_class"], average="weighted", zero_division=0)
recall_non = recall_score(df["gold_class"], df["pred_class"], average="weighted", zero_division=0)
f1_non = f1_score(df["gold_class"], df["pred_class"], average="weighted", zero_division=0)

summary_non = pd.DataFrame([{
    "Accuracy": accuracy_non,
    "Precision": precision_non,
    "Recall": recall_non,
    "F1-Score": f1_non
}])

print("\n=== Evaluation Summary ===")
display(summary_non.style.format("{:.4f}"))

# PERBANDINGAN RAG DENGAN NON RAG

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

precision = precision_score(y_true, y_pred, average="weighted", zero_division=0)
recall = recall_score(y_true, y_pred, average="weighted", zero_division=0)
f1 = f1_score(y_true, y_pred, average="weighted", zero_division=0)

summary_compare = pd.DataFrame([
    {
        "Model": "NON-RAG",
        "Accuracy": accuracy_non,
        "Precision": precision_non,
        "Recall": recall_non,
        "F1-Score": f1_non
    },
    {
        "Model": "RAG",
        "Accuracy": acc,
        "Precision": precision,
        "Recall": recall,
        "F1-Score": f1
    }
])

print("\n=== Perbandingan Performa Model ===")

display(summary_compare.style.format({
    "Accuracy": "{:.4f}",
    "Precision": "{:.4f}",
    "Recall": "{:.4f}",
    "F1-Score": "{:.4f}"
}))

# EVALUASI COHEN KAPPA SCORE

In [ ]:
from sklearn.metrics import cohen_kappa_score

bin_df = df.copy()

bin_df["gold_bool"] = bin_df["gold_class"].map({
    "FAKTA": True,
    "MITOS": False
})

bin_df["pred_bool"] = bin_df["pred_class"].map({
    "FAKTA": True,
    "MITOS": False
})

bin_df = bin_df.dropna(subset=["gold_bool","pred_bool"])

y_true_non = bin_df["gold_bool"].astype(bool).values
y_pred_non = bin_df["pred_bool"].astype(bool).values

In [ ]:
from sklearn.metrics import cohen_kappa_score

kappa_rag = cohen_kappa_score(y_true, y_pred)
kappa_non = cohen_kappa_score(y_true_non, y_pred_non)

print(f"\nCohen’s Kappa (RAG): {kappa_rag:.4f}")
print(f"Cohen’s Kappa (NON-RAG): {kappa_non:.4f}")

summary_compare = pd.DataFrame([
    {"Model": "NON-RAG", "Accuracy": accuracy_non, "Kappa": kappa_non},
    {"Model": "RAG", "Accuracy": acc, "Kappa": kappa_rag},
])

print("\n=== Perbandingan Akurasi dan Cohen’s Kappa ===")
display(summary_compare.style.format({"Accuracy": "{:.4f}", "Kappa": "{:.4f}"}))

# HALAMAN

In [ ]:
import gradio as gr
import pandas as pd
import numpy as np
import re

def process_and_index_new_doc(doc_id: str, evidence_text: str):
    """
    Pipeline otomatis untuk klaim baru:
      1. Token-based chunking (MIN=110, MAX=180, OVERLAP=30)
      2. Encode chunk → embedding (same model: all-MiniLM-L6-v2)
      3. Tambahkan ke df_chunks global & FAISS IndexFlatIP
    """
    global df_chunks, chunk_emb, index

    new_chunks = chunk_by_tokens_strict(evidence_text)

    if not new_chunks:
        return 0

    new_rows = [
        {
            "doc_id":     str(doc_id),
            "chunk_id":   f"{doc_id}::c{j}",
            "chunk_text": ch,
        }
        for j, ch in enumerate(new_chunks)
    ]
    new_df = pd.DataFrame(new_rows)

    new_emb = enc_passages(new_df["chunk_text"].tolist())   # ndarray (n, dim)

    df_chunks  = pd.concat([df_chunks, new_df], ignore_index=True)
    chunk_emb  = np.vstack([chunk_emb, new_emb])
    index.add(new_emb)   # IndexFlatIP mendukung incremental add

    n = len(new_rows)
    print(f"[index] +{n} chunk(s) | doc_id={doc_id} | total chunks={len(df_chunks)}")
    return n

def add_claim(claim_text, evidence_text, label_text):
    global df, id2gold

    claim_text   = str(claim_text).strip()
    evidence_text = str(evidence_text).strip()

    if not claim_text or not evidence_text:
        return (
            "❌ Claim dan Evidence tidak boleh kosong.",
            df.tail(5)[["id","claim","label"]],
            pd.DataFrame()
        )

    new_id = str(len(df) + 1)

    new_row = {
        "id":       new_id,
        "claim":    claim_text,
        "evidence": evidence_text,
        "label":    label_text.lower(),
    }

    df = pd.concat([df, pd.DataFrame([new_row])], ignore_index=True)
    df.to_excel(DATA_PATH, index=False)

    id2gold[new_id] = label_text.lower() == "true"

    try:
        n_chunks = process_and_index_new_doc(new_id, evidence_text)
        index_msg = f"✅ {n_chunks} chunk(s) berhasil diindeks ke FAISS."
    except Exception as e:
        index_msg = f"⚠️  Data tersimpan, tapi indexing gagal: {e}"

    status_msg = (
        f"✔  Data berhasil ditambahkan!\n"
        f"ID       : {new_id}\n"
        f"Label    : {label_text}\n"
        f"Indexing : {index_msg}"
    )

    return (
        status_msg,
        df.tail(5)[["id","claim","label"]],
        df.tail(1)[["id","claim","evidence","label"]],
    )

# CHATBOT RAG

def chatbot_answer(question, history):
    question = question.strip()
    if not question:
        history.append(("", "❓ Pertanyaan tidak boleh kosong."))
        return history, history

    candidates = retrieve_candidates(question)
    picked     = rerank_passages(question, candidates)
    evidences  = [p for (p, _, _) in picked]

    if is_yesno(question):
        fm_label, yn_raw, explanation = answer_yesno_with_explanation(
            question, evidences, picked
        )
        final_answer = f"**{fm_label}**\n\n{explanation}"
    else:
        wh = answer_wh(question, evidences, max_words=12)
        final_answer = f"**Jawaban:** {wh}"

    history.append((question, final_answer))
    return history, history

# LOAD DATA

DATA_PATH = "/content/pubhealth_biomed_only.xlsx"
df_raw    = pd.read_excel(DATA_PATH)

df = pd.DataFrame({
    "id":       df_raw.get("id", pd.Series(range(len(df_raw)))).astype(str),
    "claim":    df_raw["claim"]    if "claim"    in df_raw.columns else df_raw["text_1"],
    "evidence": df_raw["evidence"] if "evidence" in df_raw.columns else df_raw["text_2"],
    "label":    df_raw["label"].astype(str).str.lower(),
})
df = df.dropna(subset=["claim", "evidence"])
df = df[df["claim"].astype(str).str.strip()    != ""]
df = df[df["evidence"].astype(str).str.strip() != ""]

# Pastikan id2gold tersedia (dibuat dari df asli)
if "id2gold" not in dir():
    id2gold = {
        str(r["id"]): str(r["label"]).lower() == "true"
        for _, r in df.iterrows()
    }

# GRADIO UI

CSS = """
/* ── global ── */
body, .gradio-container {
    font-family: 'IBM Plex Mono', monospace;
    background: #f5f5f0 !important;
    color: #1a1a1a;
}
h1, h2, h3, .gr-markdown h1 { color: #0a7c52; letter-spacing: -0.5px; }
p, .gr-markdown p { color: #3a3a3a !important; }

/* ── panels ── */
.gr-box, .gr-panel, .gr-form, .gr-block {
    background: #ffffff !important;
    border: 1px solid #dde0d9 !important;
    border-radius: 8px !important;
    box-shadow: 0 1px 4px rgba(0,0,0,0.06) !important;
}

/* ── inputs ── */
textarea, input[type=text], .gr-input {
    background: #fafaf8 !important;
    color: #1a1a1a !important;
    border: 1px solid #c8cfc4 !important;
    border-radius: 5px !important;
    font-family: 'IBM Plex Mono', monospace !important;
}
textarea:focus, input[type=text]:focus {
    border-color: #0a7c52 !important;
    outline: none !important;
    box-shadow: 0 0 0 3px rgba(10,124,82,0.10) !important;
}

/* ── buttons ── */
.gr-button-primary {
    background: #0a7c52 !important;
    color: #ffffff !important;
    font-weight: 700 !important;
    border: none !important;
    border-radius: 5px !important;
    letter-spacing: 0.4px;
    transition: background 0.2s, box-shadow 0.2s;
    box-shadow: 0 2px 6px rgba(10,124,82,0.18) !important;
}
.gr-button-primary:hover { background: #086b45 !important; box-shadow: 0 3px 10px rgba(10,124,82,0.28) !important; }
.gr-button-secondary {
    background: #ffffff !important;
    color: #555 !important;
    border: 1px solid #c8cfc4 !important;
    border-radius: 5px !important;
}
.gr-button-secondary:hover { border-color: #0a7c52 !important; color: #0a7c52 !important; }

/* ── chatbot bubbles ── */
.message.user { background: #edf7f2 !important; border-left: 3px solid #0a7c52; color: #1a1a1a !important; }
.message.bot  { background: #ffffff !important; border-left: 3px solid #b0c4bb; color: #1a1a1a !important; }

/* ── dataframe ── */
table { border-collapse: collapse; width: 100%; }
th { background: #edf7f2; color: #0a7c52; font-size: 11px; text-transform: uppercase; letter-spacing: 1px; border-bottom: 2px solid #b8dccf; }
td { color: #3a3a3a; font-size: 12px; border-bottom: 1px solid #eee; }
tr:hover td { background: #f5fbf8; }

/* ── tab ── */
.tab-nav button { color: #777 !important; font-family: 'IBM Plex Mono', monospace !important; background: transparent !important; }
.tab-nav button.selected { color: #0a7c52 !important; border-bottom: 2px solid #0a7c52 !important; font-weight: 700 !important; }
.tab-nav { border-bottom: 1px solid #dde0d9 !important; }

/* ── label/dropdown ── */
label { color: #555 !important; font-size: 12px !important; letter-spacing: 0.5px; font-weight: 600 !important; }
.gr-dropdown, select { background: #fafaf8 !important; color: #1a1a1a !important; border: 1px solid #c8cfc4 !important; border-radius: 5px !important; }

/* ── status textbox ── */
.gr-textbox[readonly], textarea[readonly] {
    background: #f0f7f4 !important;
    border-color: #b8dccf !important;
    color: #2a5a42 !important;
}
"""

with gr.Blocks(css=CSS, title="PubHealth RAG") as app:

    gr.HTML("""
    <link href="https://fonts.googleapis.com/css2?family=IBM+Plex+Mono:wght@400;700&display=swap" rel="stylesheet">
    <div style="padding:28px 0 12px 0; border-bottom:2px solid #b8dccf; margin-bottom:20px;
                display:flex; align-items:baseline; gap:14px;">
        <span style="font-family:'IBM Plex Mono',monospace; font-size:22px; font-weight:700;
                     color:#0a7c52; letter-spacing:-0.5px;">&#11041; PUBHEALTH RAG SYSTEM</span>
        <span style="font-family:'IBM Plex Mono',monospace; font-size:12px; color:#999;">
            all-MiniLM-L6-v2 &middot; Flan-T5-base &middot; FAISS IndexFlatIP
        </span>
    </div>
    """)

    with gr.Tabs():

        # Tambah Data
        with gr.Tab("➕  Tambah Data"):

            gr.Markdown(
                "Klaim baru akan otomatis melalui **chunking → embedding → FAISS indexing** "
                "sehingga langsung bisa di-retrieve oleh chatbot."
            )

            with gr.Row():
                with gr.Column(scale=2):
                    claim_in    = gr.Textbox(label="CLAIM",    lines=3,
                                             placeholder="Masukkan klaim kesehatan…")
                    evidence_in = gr.Textbox(label="EVIDENCE", lines=5,
                                             placeholder="Masukkan teks bukti / artikel ilmiah…")
                    label_in    = gr.Dropdown(
                        choices=["true", "false", "mixture", "unproven"],
                        value="unproven",
                        label="LABEL"
                    )
                    add_btn = gr.Button("Tambah & Indeks", variant="primary")

                with gr.Column(scale=3):
                    status_out = gr.Textbox(
                        label="STATUS",
                        lines=5,
                        interactive=False,
                        placeholder="Hasil proses muncul di sini…"
                    )
                    gr.Markdown("**5 Data Terakhir**")
                    last_rows = gr.Dataframe(
                        headers=["id","claim","label"],
                        interactive=False,
                        wrap=True
                    )
                    gr.Markdown("**Data Baru Ditambahkan**")
                    new_rows = gr.Dataframe(
                        headers=["id","claim","evidence","label"],
                        interactive=False,
                        wrap=True
                    )

            add_btn.click(
                add_claim,
                inputs=[claim_in, evidence_in, label_in],
                outputs=[status_out, last_rows, new_rows]
            ).then(lambda: gr.update(value=""), None, claim_in
            ).then(lambda: gr.update(value=""), None, evidence_in)

        # Chatbot RAG
        with gr.Tab("💬  Chatbot RAG"):

            gr.Markdown(
                "Tanyakan klaim kesehatan. Pertanyaan **yes/no** → FAKTA/MITOS + bukti. "
                "Pertanyaan **wh-** → jawaban singkat."
            )

            chatbot   = gr.Chatbot(height=420, bubble_full_width=False)
            with gr.Row():
                msg       = gr.Textbox(
                    label="",
                    placeholder="Ketik pertanyaan lalu tekan Enter atau klik Kirim…",
                    lines=2,
                    scale=5
                )
                send_btn  = gr.Button("Kirim ↗", variant="primary", scale=1)
            clear_btn = gr.Button("🗑  Hapus Percakapan", variant="secondary")

            state = gr.State([])

            send_btn.click(
                chatbot_answer,
                inputs=[msg, state],
                outputs=[chatbot, state]
            ).then(lambda: gr.update(value=""), None, msg)

            msg.submit(
                chatbot_answer,
                inputs=[msg, state],
                outputs=[chatbot, state]
            ).then(lambda: gr.update(value=""), None, msg)

            clear_btn.click(lambda: ([], []), None, [chatbot, state])

app.launch(share=True, debug=False)

In [ ]:
from transformers import AutoTokenizer
tok_ref = AutoTokenizer.from_pretrained("google/flan-t5-base")

def count_tokens(text):
    return len(tok_ref(str(text), truncation=False).input_ids)

df["tok_claim"] = df["claim"].apply(count_tokens)
df["tok_evidence"] = df["evidence"].apply(count_tokens)

print("Ringkasan token claim:")
display(df["tok_claim"].describe())

print("\nRingkasan token evidence:")
display(df["tok_evidence"].describe())

print("\nContoh 10 baris hasil tokenisasi:")
display(df[["id","tok_claim","tok_evidence","label"]].head(10))

In [ ]:
print("Embedding model:", EMB_MODEL_NAME)
print("Shape embedding (jumlah_chunk, dimensi):", chunk_emb.shape)

dim_total = chunk_emb.shape[1]

selected_dims = [0, 1, dim_total-2, dim_total-1]

emb_preview = pd.DataFrame(
    chunk_emb[:3][:, selected_dims],
    columns=[f"dim_{i}" for i in selected_dims]
)

emb_preview.insert(
    0,
    "chunk_id",
    df_chunks["chunk_id"].head(3).values
)

display(emb_preview.round(6))

In [ ]:
print("FAISS index type:", type(index))
print("Dimensi index:", index.d)
print(("Total chunks:", len(df_chunks)))
sample_q = qdf["question"].iloc[0]
scores, idxs = faiss_search_topk(sample_q, topk=5)

print("Contoh pertanyaan:", sample_q)
print("\nTop-5 hasil FAISS (score + doc_id + chunk_text ringkas):")
top5 = df_chunks.iloc[idxs][["doc_id","chunk_id","chunk_text"]].copy()
top5["score"] = scores
top5["chunk_text"] = top5["chunk_text"].astype(str).str.slice(0, 180) + "..."
display(top5[["score","doc_id","chunk_id","chunk_text"]])


In [ ]:
sample_q = qdf["question"].iloc[0]
print("Sample Question:")
print(sample_q)

candidates = retrieve_candidates(sample_q, topk=TOPK_RETRIEVE)

print("\nJumlah kandidat dari FAISS:", len(candidates))

reranked = rerank_passages(
    sample_q,
    candidates,
    topm=TOPM_CONTEXT,
    max_per_doc=2
)

df_rerank = pd.DataFrame(
    reranked,
    columns=["chunk_text", "doc_id", "rerank_score"]
)

df_rerank["chunk_text"] = (
    df_rerank["chunk_text"]
    .astype(str)
    .str.slice(0, 250) + "..."
)

print("\nTop hasil setelah Reranking (CrossEncoder):")
display(df_rerank)

In [ ]:
import matplotlib.pyplot as plt

claim_stats = df["tok_claim"].describe()

plt.figure()
bars = plt.bar(claim_stats.index.astype(str), claim_stats.values)

for i, v in enumerate(claim_stats.values):
    plt.text(i, v, f"{v:.2f}", ha='center', va='bottom')

plt.title("Ringkasan Statistik Token Claim")
plt.xlabel("Statistik")
plt.ylabel("Nilai")
plt.xticks(rotation=45)
plt.show()

evidence_stats = df["tok_evidence"].describe()

plt.figure()
bars = plt.bar(evidence_stats.index.astype(str), evidence_stats.values)

for i, v in enumerate(evidence_stats.values):
    plt.text(i, v, f"{v:.2f}", ha='center', va='bottom')

plt.title("Ringkasan Statistik Token Evidence")
plt.xlabel("Statistik")
plt.ylabel("Nilai")
plt.xticks(rotation=45)
plt.show()